In [13]:
from groundingdino.util.inference import load_model, load_image, predict, annotate
import cv2

DEVICE = "cpu"

# model = load_model("groundingdino/config/GroundingDINO_SwinT_OGC.py", "weights/groundingdino_swint_ogc.pth", device=DEVICE)

model = load_model("groundingdino/config/GroundingDINO_SwinB_cfg.py", "weights/groundingdino_swinb_cogcoor.pth", device=DEVICE)

IMAGE_PATH = "assets/nid_45.png"
TEXT_PROMPT = "head . wing . fuselage . engine . tail"
BOX_TRESHOLD = 0.3
TEXT_TRESHOLD = 0.25

image_source, image = load_image(IMAGE_PATH)

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_TRESHOLD,
    text_threshold=TEXT_TRESHOLD,
    device=DEVICE
)

annotated_frame = annotate(
    image_source=image_source,
    boxes=boxes,
    logits=logits,
    phrases=phrases
)

cv2.imwrite("annotated_image_swinb.jpg", annotated_frame)
print("Saved successfully")

final text_encoder_type: bert-base-uncased
Saved successfully


# Aircraft Dataset

In [ ]:
import os
import cv2
import torch
import json
import numpy as np
from collections import Counter
from torchvision.ops import box_convert, batched_nms
from groundingdino.util.inference import load_model, load_image, predict, annotate
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BOX_THRESHOLD = 0.28
TEXT_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.45

model = load_model(
    "groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "weights/groundingdino_swinb_cogcoor.pth",
    device=DEVICE
)

INPUT_DIR = "Aircraft"
OUTPUT_DIR = "Annotated_Aircraft_Optimized_NMS"
JSON_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "grounding_dino_aircraft.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPT_GROUPS = ["head", "swept wing", "engine", "tail . tail wing"]

image_list = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
results_list = []

for img_name in tqdm(image_list, desc="GroundingDINO Aircraft"):
    IMAGE_PATH = os.path.join(INPUT_DIR, img_name)
    image_source, image = load_image(IMAGE_PATH)
    h, w, _ = image_source.shape
    
    all_boxes, all_logits, all_phrases = [], [], []

    for prompt in PROMPT_GROUPS:
        boxes, logits, phrases = predict(model, image, prompt, BOX_THRESHOLD, TEXT_THRESHOLD, device=DEVICE)
        if len(boxes) > 0:
            all_boxes.append(boxes); all_logits.append(logits); all_phrases.extend(phrases)

    if not all_boxes:
        results_list.append({
            "image_id": img_name, "components": {}, "engine_count": 0,
            "average_confidence": 0.0, "total_detections": 0, "detections": []
        })
        cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), image_source)
        continue

    all_boxes = torch.cat(all_boxes, dim=0)
    all_logits = torch.cat(all_logits, dim=0)
    boxes_xyxy = box_convert(all_boxes, in_fmt="cxcywh", out_fmt="xyxy")

    unique_phrases = list(set(all_phrases))
    phrase_to_id = {phrase: i for i, phrase in enumerate(unique_phrases)}
    class_idxs = torch.tensor([phrase_to_id[p] for p in all_phrases], device=DEVICE)

    keep = batched_nms(boxes_xyxy, all_logits, class_idxs, NMS_IOU_THRESHOLD)
    
    final_boxes, final_logits = all_boxes[keep], all_logits[keep]
    final_phrases = [all_phrases[i] for i in keep]
    final_boxes_xyxy = boxes_xyxy[keep]

    img_detections = []
    for i in range(len(final_phrases)):
        x1, y1, x2, y2 = final_boxes_xyxy[i].tolist()
        img_detections.append({
            "class": final_phrases[i],
            "confidence": round(float(final_logits[i]), 4),
            "bbox": [round(x1 * w, 2), round(y1 * h, 2), round(x2 * w, 2), round(y2 * h, 2)]
        })

    phrase_counts = dict(Counter(final_phrases))
    results_list.append({
        "image_id": img_name,
        "components": phrase_counts,
        "engine_count": phrase_counts.get("engine", 0),
        "average_confidence": round(float(torch.mean(final_logits)), 2),
        "total_detections": len(img_detections),
        "detections": img_detections
    })

    annotated_frame = annotate(image_source=image_source, boxes=final_boxes, logits=final_logits, phrases=final_phrases)
    cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), annotated_frame)

final_output = {
    "model": "GroundingDINO",
    "confidence_threshold": BOX_THRESHOLD,
    "total_images": len(image_list),
    "total_detections": sum(r["total_detections"] for r in results_list),
    "results": results_list
}

with open(JSON_OUTPUT_FILE, "w") as f:
    json.dump(final_output, f, indent=2)

final text_encoder_type: bert-base-uncased


Processing images: 100%|██████████| 70/70 [1:16:44<00:00, 65.77s/it, file=nid_3.png]         

All images processed. Summary saved to Annotated_Aircraft_Optimized_NMS/aircraft_detection_summary.json


# Car Dataset

In [4]:
import os
import cv2
import torch
import json
import numpy as np
from collections import Counter
from torchvision.ops import box_convert, batched_nms
from groundingdino.util.inference import load_model, load_image, predict, annotate
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIDENCE_THRESHOLD = 0.30 # BOX_THRESHOLD

model = load_model(
    "groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "weights/groundingdino_swinb_cogcoor.pth",
    device=DEVICE
)

INPUT_DIR = "car_hidream"
OUTPUT_DIR = "Annotated_Car_Optimized_NMS" 
os.makedirs(OUTPUT_DIR, exist_ok=True)
JSON_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "grounding_dino_car.json")

PROMPT_GROUPS = [
    "windshield . headlights . bonnet (front hood)",
    "wheels . doors . car fender . side windows . rearview mirrors",
    "trunk (rear compartment) . taillights . roof",
    "front bumper . rear bumper . grille (front panel)"
]

TEXT_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.45

image_list = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
results_list = []

for img_name in tqdm(image_list, desc="GroundingDINO Car"):
    IMAGE_PATH = os.path.join(INPUT_DIR, img_name)
    image_source, image = load_image(IMAGE_PATH)
    all_boxes, all_logits, all_phrases = [], [], []

    for prompt in PROMPT_GROUPS:
        boxes, logits, phrases = predict(model, image, prompt, CONFIDENCE_THRESHOLD, TEXT_THRESHOLD, device=DEVICE)
        if len(boxes) > 0:
            all_boxes.append(boxes); all_logits.append(logits); all_phrases.extend(phrases)

    if not all_boxes:
        results_list.append({
            "image_id": img_name, "components": {}, "wheel_count": 0,
            "average_confidence": 0.0, "total_detections": 0, "detections": []
        })
        cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), image_source)
        continue

    all_boxes = torch.cat(all_boxes, dim=0)
    all_logits = torch.cat(all_logits, dim=0)
    boxes_xyxy = box_convert(all_boxes, in_fmt="cxcywh", out_fmt="xyxy")
    unique_phrases = list(set(all_phrases))
    phrase_to_id = {phrase: i for i, phrase in enumerate(unique_phrases)}
    class_idxs = torch.tensor([phrase_to_id[p] for p in all_phrases], device=DEVICE)
    keep = batched_nms(boxes_xyxy, all_logits, class_idxs, NMS_IOU_THRESHOLD)
    
    final_boxes, final_logits = all_boxes[keep], all_logits[keep]
    final_phrases = [all_phrases[i] for i in keep]
    final_boxes_xyxy = boxes_xyxy[keep]

    img_detections = []
    for i in range(len(final_phrases)):
        img_detections.append({
            "class": final_phrases[i],
            "confidence": round(float(final_logits[i]), 4),
            "bbox": [round(float(c), 2) for c in final_boxes_xyxy[i].tolist()]
        })

    phrase_counts = dict(Counter(final_phrases))
    results_list.append({
        "image_id": img_name,
        "components": phrase_counts,
        "wheel_count": phrase_counts.get("wheels", 0),
        "average_confidence": round(float(torch.mean(final_logits)), 2),
        "total_detections": len(img_detections),
        "detections": img_detections
    })
    annotated_frame = annotate(image_source=image_source, boxes=final_boxes, logits=final_logits, phrases=final_phrases)
    cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), annotated_frame)

final_output = {
    "model": "GroundingDINO",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "total_images": len(image_list),
    "total_detections": sum(r["total_detections"] for r in results_list),
    "results": results_list
}
with open(JSON_OUTPUT_FILE, "w") as f:
    json.dump(final_output, f, indent=2)

final text_encoder_type: bert-base-uncased


GroundingDINO Car: 100%|██████████| 70/70 [1:28:55<00:00, 76.22s/it]


# Clothing Dataset

In [2]:
import os
import cv2
import torch
import torchvision
import json
from collections import Counter
from torchvision.ops import box_convert
from groundingdino.util.inference import load_model, load_image, predict, annotate
from tqdm import tqdm

# Auto-detect Hardware
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Model
model = load_model(
    "groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "weights/groundingdino_swinb_cogcoor.pth",
    device=DEVICE
)

INPUT_DIR = "clothing_Dataset_Kaggle"
OUTPUT_DIR = "Annotated_clothing_Dataset_Kaggle_NMS_Filtered"
JSON_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "clothing_detection_summary.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPT_GROUPS =[
    # Group 1: Upper body main structure
    "torso . neckline . shirt collar . shoulder seam",
    
    # Group 2: Arms and openings
    "left sleeve . right sleeve . sleeve cuff . armhole",
    
    # Group 3: Lower body
    "left pant leg . right pant leg . pant's crotch",
    
    # Group 4: Edges and fasteners
    "waistline . hem . zipper"
]

BOX_THRESHOLD = 0.30
TEXT_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.5

image_list =[
    img for img in os.listdir(INPUT_DIR)
    if img.lower().endswith((".jpg", ".png", ".jpeg"))
]

# Dictionary JSON data
detection_summary = {}

progress_bar = tqdm(image_list, desc="Processing images")

for img_name in progress_bar:
    progress_bar.set_postfix(file=img_name)
    IMAGE_PATH = os.path.join(INPUT_DIR, img_name)
    
    image_source, image = load_image(IMAGE_PATH)
    
    all_boxes =[]
    all_logits = []
    all_phrases =[]
    
    for prompt in PROMPT_GROUPS:
        boxes, logits, phrases = predict(
            model=model,
            image=image,
            caption=prompt,
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
            device=DEVICE
        )
        
        if len(boxes) > 0:
            all_boxes.append(boxes)
            all_logits.append(logits)
            all_phrases.extend(phrases)
            
    if len(all_boxes) == 0:
        cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), image_source)
        detection_summary[img_name] = {} # Record empty detection
        continue

    all_boxes = torch.cat(all_boxes, dim=0)
    all_logits = torch.cat(all_logits, dim=0)
    
    boxes_xyxy = box_convert(all_boxes, in_fmt="cxcywh", out_fmt="xyxy")
    keep_indices = torchvision.ops.nms(boxes_xyxy, all_logits, NMS_IOU_THRESHOLD)
    
    final_boxes = all_boxes[keep_indices]
    final_logits = all_logits[keep_indices]
    final_phrases = [all_phrases[i] for i in keep_indices]

    # save to dictionary
    phrase_counts = dict(Counter(final_phrases))
    detection_summary[img_name] = phrase_counts

    annotated_frame = annotate(
        image_source=image_source,
        boxes=final_boxes,
        logits=final_logits,
        phrases=final_phrases
    )
    
    
    output_path = os.path.join(OUTPUT_DIR, img_name)
    cv2.imwrite(output_path, annotated_frame)
    
# Save the dictionary to a JSON file
with open(JSON_OUTPUT_FILE, "w") as json_file:
    json.dump(detection_summary, json_file, indent=4)

print(f"All images processed. Summary saved to {JSON_OUTPUT_FILE}")

final text_encoder_type: bert-base-uncased


Processing images: 100%|██████████| 36/36 [29:30<00:00, 49.17s/it, file=1_620176d9-6b63-4f8a-b8f9-7f33b37bcf2d_jpg.rf.c6337e9732058f8d6653d0f9f046bace.jpg]            

All images processed. Summary saved to Annotated_clothing_Dataset_Kaggle_NMS_Filtered/clothing_detection_summary.json


# Animal Dataset

In [ ]:
import os
import cv2
import torch
import json
import tempfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from torchvision.ops import box_convert
from groundingdino.util.inference import load_model, load_image, predict, annotate
from tqdm import tqdm

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

model = load_model(
    "groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "weights/groundingdino_swinb_cogcoor.pth",
    device=DEVICE
)

INPUT_DIR = "generated_samples_for_animals_exluded_farfella"
OUTPUT_DIR = "generated_samples_for_animals_exluded_farfella_Optimized_NMS"
JSON_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "animal_summary_excluded_farfella.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

ALL_KEYS = ["head", "eye", "leg", "tail", "torso", "beak", "wing"]
CLASSIFICATION_PROMPT = "dog . elephant . chicken . cat . cow . sheep . squirrel"

CLASS_PART_PROMPTS = {
    "dog": ["head . eye", "leg . tail . animal torso"],
    "elephant": ["animal head . animal eye", "animal leg . animal tail . animal torso"],
    "chicken": ["animal head . animal eye . bird beak . bird wings", "animal leg . animal tail . animal torso"],
    "cat": ["animal head . animal eye", "animal leg . animal tail . animal torso"],
    "cow": ["animal head . animal eye", "animal leg . animal tail . animal torso"],
    "sheep": ["animal head . animal eye", "animal leg . animal tail . animal torso"],
    "squirrel": ["animal head . animal eye", "animal leg . animal tail . animal torso"],
}

FALLBACK_PART_PROMPTS = ["head . eye . bird beak", "leg . bird wings", "animal tail . animal torso"]

BOX_THRESHOLD = 0.28
TEXT_THRESHOLD = 0.25
CLASSIFY_THRESHOLD = 0.3
SOFT_NMS_SIGMA = 0.4
SOFT_NMS_SCORE_THRESH = 0.20
INFERENCE_SCALES = [1.0, 1.5]

def classify_animal(image_source, image):
    boxes, logits, phrases = predict(model=model, image=image, caption=CLASSIFICATION_PROMPT, box_threshold=CLASSIFY_THRESHOLD, text_threshold=TEXT_THRESHOLD, device=DEVICE)
    if len(logits) == 0: return None, 0.0
    best_idx = int(logits.argmax().item())
    best_class = phrases[best_idx].strip().lower()
    best_conf = round(logits[best_idx].item(), 4)
    for known_class in CLASS_PART_PROMPTS:
        if known_class in best_class: return known_class, best_conf
    return None, best_conf

def predict_multiscale(image_source, image, prompt):
    all_boxes, all_logits, all_phrases = [], [], []
    for scale in INFERENCE_SCALES:
        if scale != 1.0:
            h, w, _ = image_source.shape
            scaled_bgr = cv2.resize(cv2.cvtColor(image_source, cv2.COLOR_RGB2BGR), (int(w * scale), int(h * scale)))
            with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
                cv2.imwrite(tmp.name, scaled_bgr)
                tmp_path = tmp.name
            _, scaled_image = load_image(tmp_path)
            os.unlink(tmp_path)
            boxes, logits, phrases = predict(model=model, image=scaled_image, caption=prompt, box_threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD, device=DEVICE)
            if len(boxes) > 0: boxes = boxes.clamp(0.0, 1.0)
        else:
            boxes, logits, phrases = predict(model=model, image=image, caption=prompt, box_threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD, device=DEVICE)
        if len(boxes) > 0:
            all_boxes.append(boxes); all_logits.append(logits); all_phrases.extend(phrases)
    return all_boxes, all_logits, all_phrases

def _iou(b1, b2):
    x1, y1 = max(b1[0].item(), b2[0].item()), max(b1[1].item(), b2[1].item())
    x2, y2 = min(b1[2].item(), b2[2].item()), min(b1[3].item(), b2[3].item())
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a1, a2 = (b1[2] - b1[0]).item() * (b1[3] - b1[1]).item(), (b2[2] - b2[0]).item() * (b2[3] - b2[1]).item()
    return inter / (a1 + a2 - inter + 1e-6)

def soft_nms(boxes_xyxy, scores, sigma=SOFT_NMS_SIGMA, score_thresh=SOFT_NMS_SCORE_THRESH):
    decay, kept = scores.clone().float(), []
    order = decay.argsort(descending=True).tolist()
    while order:
        i = order.pop(0)
        if decay[i].item() < score_thresh: break
        kept.append(i)
        remaining = []
        for j in order:
            iou = _iou(boxes_xyxy[i], boxes_xyxy[j])
            decay[j] *= torch.exp(torch.tensor(-iou ** 2 / sigma)).item()
            if decay[j].item() >= score_thresh: remaining.append(j)
        order = sorted(remaining, key=lambda x: -decay[x].item())
    return kept

image_list = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
def _load_worker(img_name): return img_name, load_image(os.path.join(INPUT_DIR, img_name))

with ThreadPoolExecutor(max_workers=4) as pool:
    loaded_images = dict(pool.map(_load_worker, image_list))

results_list = []
for img_name in tqdm(image_list, desc="Processing images"):
    image_source, image = loaded_images[img_name]
    h, w, _ = image_source.shape
    predicted_class, class_conf = classify_animal(image_source, image)
    part_prompts = CLASS_PART_PROMPTS[predicted_class] if predicted_class else FALLBACK_PART_PROMPTS
    all_boxes, all_logits, all_phrases = [], [], []
    for prompt in part_prompts:
        g_boxes, g_logits, g_phrases = predict_multiscale(image_source, image, prompt)
        all_boxes.extend(g_boxes); all_logits.extend(g_logits); all_phrases.extend(g_phrases)

    if not all_boxes:
        cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), image_source)
        results_list.append({"image_id": img_name, "components": {k: 0 for k in ALL_KEYS}, "leg_count": 0, "average_confidence": 0.0, "total_detections": 0, "detections": []})
        continue

    all_boxes, all_logits = torch.cat(all_boxes, dim=0), torch.cat(all_logits, dim=0)
    boxes_xyxy = box_convert(all_boxes, in_fmt="cxcywh", out_fmt="xyxy").clamp(0, 1)
    keep_indices = soft_nms(boxes_xyxy, all_logits)
    final_boxes, final_logits, final_boxes_xyxy = all_boxes[keep_indices], all_logits[keep_indices], boxes_xyxy[keep_indices]
    final_phrases = [all_phrases[i].replace("animal ", "").replace("butterfly ", "").strip().lower() for i in keep_indices]

    normalized = []
    for p in final_phrases:
        if "eye" in p: normalized.append("eye")
        elif "leg" in p: normalized.append("leg")
        elif "tail" in p and "wing" not in p: normalized.append("tail")
        elif "torso" in p or "body" in p: normalized.append("torso")
        elif "head" in p: normalized.append("head")
        elif "beak" in p: normalized.append("beak")
        elif "wing" in p: normalized.append("wing")

    components_fixed = {k: Counter(normalized).get(k, 0) for k in ALL_KEYS}
    detections_output = [{"class": p, "confidence": round(l.item(), 4), "bbox": [round(b[0]*w, 2), round(b[1]*h, 2), round(b[2]*w, 2), round(b[3]*h, 2)]} for b, l, p in zip(final_boxes_xyxy, final_logits, final_phrases)]

    results_list.append({
        "image_id": img_name, "components": components_fixed, "leg_count": components_fixed.get("leg", 0),
        "average_confidence": round(float(final_logits.mean().item()), 2) if len(final_logits) > 0 else 0.0,
        "total_detections": len(detections_output), "detections": detections_output
    })
    annotated = annotate(image_source=image_source, boxes=final_boxes, logits=final_logits, phrases=final_phrases)
    cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), annotated)

final_output = {
    "model": "GroundingDINO",
    "confidence_threshold": BOX_THRESHOLD,
    "total_images": len(image_list),
    "total_detections": sum(r["total_detections"] for r in results_list),
    "results": results_list
}
with open(JSON_OUTPUT_FILE, "w") as f:
    json.dump(final_output, f, indent=2)

Using device: cpu
final text_encoder_type: bert-base-uncased
Pre-loading images …
Loaded 70 images.



Processing images: 100%|██████████| 70/70 [1:25:51<00:00, 73.59s/it, file=cane_7.jpg]      


Done. Summary saved to generated_samples_for_animals_exluded_farfella_Optimized_NMS/animal_summary_excluded_farfella.json


In [ ]:
prompt_template = (
    "<image>\n"
    "You are an animal body-part detector. Carefully examine this image and count ONLY the parts that are clearly visible.\n\n"
    "Count these 7 parts:\n"
    "(a) Head - the animal head. For a clearly visible animal, this is usually 1.\n"
    "(b) Eyes - visible animal eyes. If both eyes are clearly visible, count 2. If only one eye is visible, count 1. If none are visible, count 0.\n"
    "(c) Legs - visible animal legs. Count each clearly visible leg. Some animals may have 2(if it's a bird) or 4 visible legs depending on pose and occlusion.\n"
    "(d) Tail - visible animal tail. Usually 1 if visible.\n"
    "(e) Torso - main body of the animal. Usually 1 if visible.\n"
    "(f) Beak - ONLY for birds. Count 1 if clearly visible, otherwise 0.\n"
    "(g) Wings - ONLY for birds. If both wings are visible, count 2. If only one wing is visible, count 1. If none are visible, count 0.\n\n"
    "IMPORTANT RULES:\n"
    "- Only count parts that are clearly visible in the image.\n"
    "- Do NOT assume hidden or occluded parts exist.\n"
    "- If you cannot see a component clearly, count it as 0.\n"
    "- Base all counts only on visible evidence.\n"
    "- Use the most likely visible count, not a guess.\n\n"
    "Respond EXACTLY in this format:\n"
    "(a) Head: [number]\n"
    "(b) Eyes: [number]\n"
    "(c) Legs: [number]\n"
    "(d) Tail: [number]\n"
    "(e) Torso: [number]\n"
    "(f) Beak: [number]\n"
    "(g) Wings: [number]\n"
    "Overall confidence: [0.0-1.0]"
)